<a href="https://colab.research.google.com/github/makxxic/AI-for-Science-and-Engineering/blob/main/Week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Problem and Intended AI/ML Outcome

## Project Problem

Loan default is a significant challenge for financial institutions because borrowers who fail to repay their loans can result in financial losses and increased credit risk. The project focuses on identifying factors associated with loan default and using borrower and loan information to assess the likelihood that a borrower will default.

## Intended AI/ML Outcome

The intended outcome is to develop a **machine learning classification model** that can predict whether a borrower is likely to **default or not default** on a loan based on relevant financial and demographic characteristics.

The model will learn patterns from historical loan data and use these patterns to classify new borrowers into two categories:

- **1 – Default:** The borrower is likely to default on the loan.
- **0 – Non-default:** The borrower is likely to repay the loan.

The goal is to provide a data driven approach that can help financial institutions identify potentially high risk loans and make more informed lending and risk management decisions.

# Import necessary libraries


In [1]:
import kagglehub as kh # Import kagglehub for downloading datasets
import pandas as pd # Import pandas for data manipulation and analysis
import os # Import os for interacting with the operating system
import numpy as np # Import numpy for numerical operations

#  Download Dataset
This cell uses kagglehub to download a specific dataset. It then prints the path where the dataset files are located and lists the contents of that directory to confirm the downloaded files.

In [2]:
# Download the Loan Default Prediction dataset

path = kh.dataset_download("hemanthsai7/loandefault")

print("Path to dataset files:", path) # Print the local path where the dataset is stored
os.listdir(path) # List the files available in the downloaded dataset directory

Using Colab cache for faster access to the 'loandefault' dataset.
Path to dataset files: /kaggle/input/loandefault


['train.csv', 'test.csv']

# Load Data into DataFrame
This cell reads the downloaded CSV file into a pandas DataFrame. The df.head() method is then used to display the first few rows of the DataFrame, providing a quick overview of the data structure and content.

In [3]:
df = pd.read_csv(path +'/train.csv') # Read the CSV file into a pandas DataFrame

In [4]:
# Confirm dimensions
print("Dataset shape:", df.shape)


# Display the first 5 rows transposed
display(df.head().T)

Dataset shape: (67463, 35)


,0,1,2,3,4
ID,65087372,1450153,1969101,6651430,14354669
Loan Amount,10000,3609,28276,11170,16890
Funded Amount,32236,11940,9311,6954,13226
Funded Amount Investor,12329.36286,12191.99692,21603.22455,17877.15585,13539.92667
Term,59,59,59,59,59
Batch Enrolled,BAT2522922,BAT1586599,BAT2136391,BAT2428731,BAT5341619
Interest Rate,11.135007,12.237563,12.545884,16.731201,15.0083
Grade,B,C,F,C,C
Sub Grade,C4,D3,D4,C3,D4
Employment Duration,MORTGAGE,RENT,MORTGAGE,MORTGAGE,MORTGAGE


## Check data types

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67463 entries, 0 to 67462
Data columns (total 35 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   ID                            67463 non-null  int64  
 1   Loan Amount                   67463 non-null  int64  
 2   Funded Amount                 67463 non-null  int64  
 3   Funded Amount Investor        67463 non-null  float64
 4   Term                          67463 non-null  int64  
 5   Batch Enrolled                67463 non-null  object 
 6   Interest Rate                 67463 non-null  float64
 7   Grade                         67463 non-null  object 
 8   Sub Grade                     67463 non-null  object 
 9   Employment Duration           67463 non-null  object 
 10  Home Ownership                67463 non-null  float64
 11  Verification Status           67463 non-null  object 
 12  Payment Plan                  67463 non-null  object 
 13  L

## Check missing values

In [6]:
missing = df.isnull().sum()

missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing Values': missing.values,
    'Missing %': (missing.values / len(df) * 100).round(2)
})

missing_df.sort_values('Missing %', ascending=False)

,Column,Missing Values,Missing %
0,ID,0,0.0
1,Loan Amount,0,0.0
2,Funded Amount,0,0.0
3,Funded Amount Investor,0,0.0
4,Term,0,0.0
5,Batch Enrolled,0,0.0
6,Interest Rate,0,0.0
7,Grade,0,0.0
8,Sub Grade,0,0.0
9,Employment Duration,0,0.0


## Examine the target
This tells us exactly how many borrowers are in each class.

Then we can say accurately whether the dataset has class imbalance.

In [7]:
print(df['Loan Status'].value_counts())
print(df['Loan Status'].value_counts(normalize=True) * 100)

Loan Status
0    61222
1     6241
Name: count, dtype: int64
Loan Status
0    90.749003
1     9.250997
Name: proportion, dtype: float64


## Examine every categorical column

In [8]:
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(20))


--- Batch Enrolled ---
Batch Enrolled
BAT3873588    3626
BAT1586599    3142
BAT1104812    2996
BAT2252229    2557
BAT2803411    2425
BAT1780517    2403
BAT1184694    2298
BAT2078974    2290
BAT2575549    2257
BAT4694572    2248
BAT4271519    2054
BAT2558388    1963
BAT3193689    1864
BAT1930365    1844
BAT2136391    1790
BAT2333412    1775
BAT3726927    1774
BAT4136152    1766
BAT5341619    1717
BAT5525466    1709
Name: count, dtype: int64

--- Grade ---
Grade
C    19085
B    18742
A    12055
D     8259
E     6446
F     2246
G      630
Name: count, dtype: int64

--- Sub Grade ---
Sub Grade
B4    4462
C1    4188
B3    3999
A5    3540
B2    3520
B5    3408
D1    3304
C4    3250
C2    3219
C3    3121
B1    2924
C5    2472
A4    2264
D4    2050
D2    1963
D5    1952
A2    1837
D3    1824
E2    1746
A3    1685
Name: count, dtype: int64

--- Employment Duration ---
Employment Duration
MORTGAGE    36351
RENT        24150
OWN          6962
Name: count, dtype: int64

--- Verification Status --

In [9]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

ID: 67463 unique values
Loan Amount: 27525 unique values
Funded Amount: 24548 unique values
Funded Amount Investor: 67441 unique values
Term: 3 unique values
Batch Enrolled: 41 unique values
Interest Rate: 67448 unique values
Grade: 7 unique values
Sub Grade: 35 unique values
Employment Duration: 3 unique values
Home Ownership: 67454 unique values
Verification Status: 3 unique values
Payment Plan: 1 unique values
Loan Title: 109 unique values
Debit to Income: 67454 unique values
Delinquency - two years: 9 unique values
Inquires - six months: 6 unique values
Open Account: 36 unique values
Public Record: 5 unique values
Revolving Balance: 20582 unique values
Revolving Utilities: 67458 unique values
Total Accounts: 69 unique values
Initial List Status: 2 unique values
Total Received Interest: 67451 unique values
Total Received Late Fee: 67380 unique values
Recoveries: 67387 unique values
Collection Recovery Fee: 67313 unique values
Collection 12 months Medical: 2 unique values
Application

In [10]:
test_df = pd.read_csv(path + '/test.csv')

print("Train shape:", df.shape)
print("Test shape:", test_df.shape)

print("\nColumns only in train:")
print(set(df.columns) - set(test_df.columns))

print("\nColumns only in test:")
print(set(test_df.columns) - set(df.columns))

Train shape: (67463, 35)
Test shape: (28913, 35)

Columns only in train:
set()

Columns only in test:
set()


In [11]:
print(test_df['Loan Status'].value_counts())
print(test_df['Loan Status'].value_counts(normalize=True) * 100)

Series([], Name: count, dtype: int64)
Series([], Name: proportion, dtype: float64)


## Dataset quality

Training data has:

67,463 rows
35 columns
No missing values
Target: Loan Status
0 = 90.75%
1 = 9.25%

So there is a clear class imbalance: roughly 1 default for every 10.8 non-default cases.

This is important because a model could achieve about 91% accuracy simply by predicting 0 for almost everyone.

## Target Variable and Class Distribution

The target variable for this project is `Loan Status`, which represents whether a loan falls into the default class.

The training dataset contains 67,463 observations. There are 61,222 observations with `Loan Status = 0` and 6,241 observations with `Loan Status = 1`.

This corresponds to approximately 90.75% class 0 and 9.25% class 1. Therefore, the target variable is imbalanced, with substantially fewer observations in class 1.

This class imbalance needs to be considered when developing and evaluating the machine learning model. Accuracy alone may be misleading because a model that predicts the majority class for most observations could achieve high accuracy while performing poorly at identifying borrowers in the minority class. Therefore, metrics such as precision, recall, F1-score and ROC-AUC will also be considered.

## Numerical features
* Loan Amount
* Funded Amount
* Funded Amount Investor
* Term
* Interest Rate
* Home Ownership
* Debit to Income
* Delinquency - two years
* Inquires - six months
* Open Account
* Public Record
* Revolving Balance
* Revolving Utilities
* Total Accounts
* Total Received Interest
* Total Received Late Fee
* Recoveries
* Collection Recovery Fee
* Collection 12 months Medical
* Last week Pay
* Total Collection Amount
* Total Current Balance
* Total Revolving Credit Limit
## Categorical features
* Batch Enrolled
* Grade
* Sub Grade
* Employment Duration
* Verification Status
* Loan Title
* Initial List Status
* Application Type
## Exclude
* ID
* Payment Plan
* Accounts Delinquent
* Loan Status

`Loan Status` is the target, so obviously it isn't an input.

ID, Payment Plan, and Accounts Delinquent should be excluded because:

`ID` uniquely identifies each observation.

`Payment Plan` has only one value.

`Accounts Delinquent` has only one value.

## Potential features

Borrower, loan, credit and financial characteristics such as:

* Loan Amount
* Funded Amount
* Term
* Interest Rate
* Grade
* Sub Grade
* Employment Duration
* Home Ownership
* Verification Status
* Debit to Income
* nDelinquencies
* Inquiries
* Open Accounts
* Public Records
* Revolving Balance
* Revolving Utilities
* Total Accounts
* Credit-related variables
etc.
## Target

`Loan Status`

The target is what the model is ultimately trying to predict.

## Identifier

`ID`

This should not be treated as a predictive feature.

## Potential leakage variables

We'll investigate:

* Total Received Interest
* Total Received Late Fee
* Recoveries
* Collection Recovery Fee
* Total Collection Amount
* Total Current Balance
* Last week Pay



## Proposed Machine Learning Workflow

```text
┌─────────────────────────┐
│    Raw Loan Dataset     │
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│    Data Inspection      │
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│ Identify Target Variable│
│    (Loan Status)        │
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│   Data Quality Checks   │
│ Missing values, types,  │
│ duplicates, imbalance   │
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│ Remove Identifier &     │
│ Constant Variables     │
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│ Investigate Potential   │
│     Data Leakage        │
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│    Feature Selection    │
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│  Categorical Encoding   │
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│ Train / Validation Split│
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│ Handle Class Imbalance  │
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│ Train Classification   │
│        Models           │
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│   Evaluate Models       │
│ Accuracy, Precision,    │
│ Recall, F1, ROC-AUC     │
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│    Compare Models       │
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│   Select Best Model     │
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│ Predict Loan Default    │
│         Risk            │
└─────────────────────────┘